In [ ]:
#------------------------------------------------------------------------------------
# This script processes geographic microdata from Dewey Data into an analysis dataset for a simple spatial DiD
# Sean McCulloch 11oct2026
# Outline:
    # 0) Clean and append together ATTOM house transaction data from Dewey; Load in two polluting plant locations
    # 1) Go to a projected CRS and draw circles around the 2 polluting plants
    # 2) Spatial join housing sales to polluting plant circles
    # 3) Calculate houses' distances to the plant 
    # 4) Aggregate joined sales to rings w/ avg prices in pre vs. post period
    
import geopandas as gpd 
import pandas as pd
import numpy as np
import glob 
import os

# Paths are relative to this notebook (src/pgm/), so the code runs wherever the repo is saved
filepath = "../../"
src = filepath + "src/"
bld = filepath + "bld/"

#---------------------------------------------------------------------------------------

In [ ]:
# 0) Clean and append together ATTOM house transaction data from Dewey

# a) Deed transfers (recorderdata)

# List out all the files in the data extract from Dewey
attom_dir = src + "raw/recorder/"
files = sorted(glob.glob(os.path.join(attom_dir, "*.csv.gz")))
print(f"Found {len(files)} files")

# Append the individual csvs into one combined dataframe
sales = pd.concat(
    (pd.read_csv(f, dtype={"PROPERTYUSESTANDARDIZED": str}, parse_dates=["RECORDINGDATE"])
     for f in files),
    ignore_index=True)

# Drop sales with missing data
# TRANSACTIONTYPE is empty in this extract
sales = sales.drop(columns="TRANSACTIONTYPE")

# 999999999 is ATTOM's placeholder for an unmatched property
sales["ATTOMID"] = sales["ATTOMID"].astype("Int64")
sales.loc[sales["ATTOMID"] == 999999999, "ATTOMID"] = pd.NA
sales

In [ ]:
# Sample selection
# Restrict to detached single family houses, arms lengths transactions (and > $10,000 and < 10 million)

# Detached single family house 
sales = sales[sales['PROPERTYUSESTANDARDIZED'] == "385"]
# Arms length
sales = sales[sales['ARMSLENGTHFLAG'] == 1]
# Sale price restrictions 
sales = sales[sales['TRANSFERAMOUNT'] >= 10000]
sales = sales[sales['TRANSFERAMOUNT'] <= 10000000]

In [ ]:
# Get sale price in real dollars
sales["sale_year"] = sales["RECORDINGDATE"].dt.year

# CPI-U (all items), monthly, averaged to annual
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL"
cpi = pd.read_csv(url, index_col=0, parse_dates=True).iloc[:, 0]
cpi_yr = cpi.groupby(cpi.index.year).mean()

# Real prices in 2025 dollars
base = cpi_yr[2025]
sales["sale_price_real"] = sales["TRANSFERAMOUNT"] * base / sales["sale_year"].map(cpi_yr)

In [3]:
os.makedirs(bld, exist_ok=True)
sales.to_csv(bld + "attom_sales.csv", index=False)
print(sales.shape)

(107709, 10)


In [ ]:
sales

In [ ]:
# b) Also, process ATTOM tax assessor data because no coordinates in the deed data
# We can match across datasets using the ID provided by ATTOM

# List files
assessor_dir = src + "raw/assessor/"
files = sorted(glob.glob(os.path.join(assessor_dir, "*.parquet")))
print(f"Found {len(files)} files")

# Combine csvs into one dataframe
parcels = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)
parcels["ATTOMID"] = pd.to_numeric(parcels["ATTOMID"]).astype("Int64")

# Drop missing or bad geocodes (zeros, -1, points far outside Rhode Island)
in_ri = (parcels["LONGITUDE"].between(-71.92, -71.10) &
         parcels["LATITUDE"].between(41.10, 42.03))
print(f"Dropping {(~in_ri).sum()} parcels with missing or out-of-state coordinates")
parcels = parcels[in_ri].reset_index(drop=True)

assert parcels["ATTOMID"].is_unique
parcels.to_parquet(bld + "attom_parcels.parquet", index=False)

# Attach coordinates to each sale by merging on the ATTOMID
sales_coordinates = sales.merge(parcels, on="ATTOMID", how="inner", validate="m:1")

sales_coordinates.to_parquet(bld + "attom_sales_geo.parquet", index=False)
print(parcels.shape, sales.shape, sales_coordinates.shape)
sales_coordinates

In [ ]:
# c) Load the two plant locations

plants_df = pd.read_csv(src + "raw/tri2005RIlist.csv")

plants_df['plantid'] = plants_df.index + 1

# Similar column names across the two geodatasets can get confusing so rename columns:
renamecoords = {'lon':'plant_lon', 'lat':'plant_lat', 'year':'plant_year'}

plants_df = plants_df.rename(columns=renamecoords)

# Thus far, everything standard tabular data cleaning
# Now, start geospatial operations 

In [7]:
#------------------------------------------------------------------------------------
# 1) Set up geodataframes, go to a projected CRS and draw circles around the 2 polluting plants

# a) Convert regular dataframes to geodataframes
# Sales
sales_coordinates_gdf = gpd.GeoDataFrame(
    sales_coordinates,
    geometry=gpd.points_from_xy(sales_coordinates["LONGITUDE"], sales_coordinates["LATITUDE"]),
    crs="EPSG:4326",   # lat/lon in degrees (WGS84)
)

# Plants
plants_gdf = gpd.GeoDataFrame(
    plants_df,
    geometry=gpd.points_from_xy(plants_df["plant_lon"], plants_df["plant_lat"]),
    crs="EPSG:4326",   # lat/lon in degrees (WGS84)
)

In [8]:
# b) Convert to a projected coordinate reference system (CRS) to get distances in meters
sales_coordinates_gdf = sales_coordinates_gdf.to_crs(epsg=5070)
plants_gdf = plants_gdf.to_crs(epsg=5070)


In [ ]:
# c) Draw 2km radii circles around the plants

plants_gdf.geometry = plants_gdf.geometry.buffer(2000)
plants_gdf

In [ ]:
#------------------------------------------------------------------------------------
# 2) Spatial join housing sales to polluting plants' circles

sales_in_rings = sales_coordinates_gdf.sjoin(plants_gdf) # Default sjoin gives an inner spatial join, see documentation for more options
sales_in_rings

In [ ]:
sales_in_rings

In [12]:
#---------
sales_in_rings.to_crs(epsg=4326).to_file(bld + "sales_in_rings.json", driver="GeoJSON")

In [ ]:
#------------------------------------------------------------------------------------
# 3) Calculate distance to the plant 

# Get associated plant's point as a geoseries
plant_pts = gpd.GeoSeries(
    gpd.points_from_xy(sales_in_rings["plant_lon"], sales_in_rings["plant_lat"]),
    index=sales_in_rings.index,
    crs="EPSG:4326",            # plant coordinates are lat/lon convert to projected CRS again
).to_crs(sales_in_rings.crs)             

plant_pts

In [ ]:
sales_in_rings["distance_to_plant"] = sales_in_rings.geometry.distance(plant_pts)
sales_in_rings

In [15]:
#---------
sales_in_rings.to_crs(epsg=4326).to_file(bld + "sales_in_rings.json", driver="GeoJSON")

# Looks correct on a map

In [ ]:
#------------------------------------------------------------------------------------
# 4) Aggregate joined sales to rings w/ avg prices in pre vs. post period

# Remember standard DiD was taking the differences using 4 averages: Treat Pre, Treat Post, Control Pre, Control Post

# Make binary indicators
# Treatment vs Control
sales_in_rings["treatment"] = (sales_in_rings["distance_to_plant"] < 1000).astype(int)

# Pre vs. Post 

sales_in_rings = sales_in_rings[sales_in_rings['sale_year'] != 2005]
sales_in_rings["post"] = (sales_in_rings["sale_year"] > sales_in_rings["plant_year"]).astype(int)

sales_in_rings["treatment"], sales_in_rings["post"] 

In [ ]:
sales_in_rings.RECORDINGDATE

In [ ]:
sales_in_rings

In [ ]:
# Get averages for the DiD 
avg = (sales_in_rings.groupby(["treatment", "post"])["sale_price_real"]
            .agg(["mean", "sem", "count"])
            .unstack("post"))
print(avg.round(0))

print(312847.0 - 298624.0 - (313488.0 - 291066.0))
print(np.sqrt(3038.0**2 + 3649.0**2 + 13738.0**2 + 6701.0**2))

# DiD indicates a $8,200 dollar depreciation in value or about 1-2%
# However, standard errors are large, so it's not significant
# There is a lot more spread in house prices for the treatment group than the control group. 
    # Why?

In [ ]:
# Can also get averages in the 4 treatment groups by plant for the 2 plants => 8 means
avg = (sales_in_rings.groupby(["plantid", "treatment", "post"])["sale_price_real"]
            .agg(["mean", "sem", "count"])
            .unstack("post"))
print(avg.round(0))

In [26]:
# Plant 1: (Treat post - treat pre) - (control post - control pre)
print(285924.0 - 262485.0 - (289572.0 - 268047.0))

# Plant 2: (Treat post - treat pre) - (control post - control pre)
print(408831.0 - 385802.0 - (408548.0 - 375220.0))

# Moderately large effect in plant 2, ~ no effect in plant 1, though again caveating we have low precision.

1914.0
-10299.0


In [19]:
#------------------------------------------------------------------------------------
# Out to file

#-----------
